In [1]:
# =========================
# BLOCK 1 — PREPROCESSING
# =========================
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

# ---- File paths ----
BASE = "/Users/adithyamadduri/Downloads/syn65414912"
ANML_PATH   = os.path.join(BASE, "OhNM2025_ROSMAP_plasma_Soma7k_protein_level_ANML_log10.csv")
SAMPLE_PATH = os.path.join(BASE, "OhNM2025_ROSMAP_plasma_Soma7k_sample_metadata.csv")

# ---- Load ----
df_levels = pd.read_csv(ANML_PATH)        # rows: projid_visit, cols: proteins
df_meta   = pd.read_csv(SAMPLE_PATH)      # contains projid_visit, projid, msex, age_at_visit, educ, apoe_genotype, Diagnosis

# ---- Sanity: keys present? ----
assert "projid_visit" in df_levels.columns, "projid_visit missing in protein matrix."
for col in ["projid_visit","projid","msex","age_at_visit","educ","apoe_genotype","Diagnosis"]:
    assert col in df_meta.columns, f"{col} missing in sample metadata."

# ---- Align on visit ----
# inner join so we only keep visits present in BOTH
df = pd.merge(df_meta, df_levels, on="projid_visit", how="inner", validate="one_to_one")
print("Aligned shape:", df.shape)

# ---- Labels (four groups) ----
df["Diagnosis"] = df["Diagnosis"].astype(str).str.strip()
valid_classes = {"MCI","NCI","AD","AD+"}
df = df[df["Diagnosis"].isin(valid_classes)].reset_index(drop=True)
print("Class counts:\n", df["Diagnosis"].value_counts())

# ---- Grouping key for leakage control ----
df["projid"] = df["projid"].astype(str)

# ---- Stratification label: Diagnosis × sex (0/1) ----
df["msex"] = df["msex"].astype(int)
df["strata"] = df["Diagnosis"] + "_" + df["msex"].astype(str)

# ---- APOE one-hot (Unknown for NaN) ----
def format_apoe(x):
    if pd.isna(x):
        return "Unknown"
    try:
        # e.g., 33.0 -> "33"
        return str(int(float(x)))
    except Exception:
        s = str(x).strip()
        return s if s else "Unknown"

df["apoe_str"] = df["apoe_genotype"].apply(format_apoe)

# scikit-learn compatibility across versions
try:
    ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
except TypeError:
    ohe = OneHotEncoder(sparse=False, handle_unknown="ignore")

apoe_ohe = ohe.fit_transform(df[["apoe_str"]])
apoe_cols = [c.replace("apoe_str_","APOE_") for c in ohe.get_feature_names_out()]
df_apoe  = pd.DataFrame(apoe_ohe, columns=apoe_cols, index=df.index)

# ---- Numeric covariates (unscaled) ----
for col in ["age_at_visit","educ"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# ---- Protein feature columns ----
protein_cols = [c for c in df_levels.columns if c != "projid_visit"]

# ---- Final feature matrix X ----
# X = pd.concat([df[["age_at_visit","educ"]], df_apoe, df[protein_cols]], axis=1)
X = pd.concat([df[["age_at_visit","educ","msex"]], df_apoe, df[protein_cols]], axis=1)
y = df["Diagnosis"].astype(str).values
groups = df["projid"].values
strata = df["strata"].values

print("X shape:", X.shape)
print("APOE levels seen:", sorted(set(df["apoe_str"])))

Aligned shape: (973, 7301)
Class counts:
 Diagnosis
NCI    507
MCI    262
AD     167
AD+     17
Name: count, dtype: int64
X shape: (953, 7299)
APOE levels seen: ['22', '23', '24', '33', '34', '44', 'Unknown']


In [2]:
# =========================
# BLOCK 2 — TRAINING
# =========================
import os
import numpy as np
import pandas as pd
from flaml import AutoML
from sklearn.metrics import roc_auc_score

# ---- Output folder ----
# This is the original folder that this code used to write to. It seems at some point I renamed this folder to _fixed. Now confirming that these results match fixed.
# out_dir = "/Users/adithyamadduri/Desktop/Projects/proteomics_LGBM(ANML+Meta)"

out_dir = "/Users/adithyamadduri/Desktop/Projects/ratios_project/Revision/Results/Proteins+Demographics"
os.makedirs(out_dir, exist_ok=True)

# ---- Helper: group-aware stratified 70/30 split (by mode of strata per subject) ----
def group_stratified_shuffle_split(df_index, strata_all, groups_all, test_size=0.30, random_state=0):
    """Return train_idx, test_idx ensuring groups stay intact and class-sex balance is approx preserved.
       Strategy: assign each group a single stratum = mode(strata) among its rows, then stratified split at group level.
    """
    rng = np.random.RandomState(random_state)
    data = pd.DataFrame({"idx": df_index, "strata": strata_all, "group": groups_all})

    # group -> mode(strata)
    grp_mode = (
        data.groupby("group")["strata"]
        .agg(lambda s: s.value_counts().idxmax())
    )
    grp_mode = grp_mode.sample(frac=1.0, random_state=random_state)  # shuffle groups

    train_groups, test_groups = [], []
    for s_val, grp_ids in grp_mode.groupby(grp_mode.values):
        g_list = list(grp_ids.index)
        rng.shuffle(g_list)
        n_test = max(1, int(round(test_size * len(g_list))))
        test_groups.extend(g_list[:n_test])
        train_groups.extend(g_list[n_test:])

    train_mask = np.isin(groups_all, train_groups)
    test_mask  = np.isin(groups_all,  test_groups)
    return np.where(train_mask)[0], np.where(test_mask)[0]

# ---- Classes & seeds ----
classes = ["MCI","NCI","AD","AD+"]     # will save AD+ as 'ADplus' in filenames
def safe_cls(c): return c.replace("+","plus").replace(" ","_").replace("/","-")

seeds = [1,2,3,4,5]

# ---- Main loop ----
for seed in seeds:
    tr_idx, te_idx = group_stratified_shuffle_split(
        df_index=np.arange(len(X)),
        strata_all=strata,
        groups_all=groups,
        test_size=0.30,
        random_state=seed,
    )

    X_train, X_test = X.iloc[tr_idx], X.iloc[te_idx]
    y_train_full    = y[tr_idx]
    y_test_full     = y[te_idx]

    print(f"\n[Seed {seed}] Train n={len(tr_idx)} | Test n={len(te_idx)} | "
          f"Groups train={len(np.unique(groups[tr_idx]))} test={len(np.unique(groups[te_idx]))}")

    for cls in classes:
        y_train = (y_train_full == cls).astype(int)
        y_test  = (y_test_full  == cls).astype(int)

        automl = AutoML()
        settings = {
            "time_budget": 100,               # seconds per class; adjust if you want longer searches
            "metric": "roc_auc",
            "task": "classification",
            "eval_method": "cv",
            "estimator_list": ["lgbm"],       # force LightGBM
            "log_file_name": os.path.join(out_dir, f"flaml_seed{seed}_{safe_cls(cls)}.log"),
            "seed": seed,
        }
        automl.fit(X_train=X_train, y_train=y_train, **settings)
        automl.pickle(os.path.join(out_dir, f"seed{seed}_{safe_cls(cls)}_automl.pkl"))

        # Predict proba for positive class
        y_score = automl.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, y_score) if (y_test.sum() > 0 and y_test.sum() < len(y_test)) else float("nan")
        print(f"  [{cls}] AUC={auc:.3f}")

        # Save per-class CSV for your existing averaging/plotting script
        out_csv = os.path.join(out_dir, f"seed{seed}_{safe_cls(cls)}.csv")
        pd.DataFrame({"y_true": y_test.astype(int), "y_score": y_score.astype(float)}).to_csv(out_csv, index=False)


[Seed 1] Train n=670 | Test n=283 | Groups train=610 test=261
[flaml.automl.logger: 02-02 22:09:02] {1752} INFO - task = classification
[flaml.automl.logger: 02-02 22:09:02] {1763} INFO - Evaluation method: cv
[flaml.automl.logger: 02-02 22:09:03] {1862} INFO - Minimizing error metric: 1-roc_auc
[flaml.automl.logger: 02-02 22:09:03] {1979} INFO - List of ML learners in AutoML Run: ['lgbm']
[flaml.automl.logger: 02-02 22:09:03] {2282} INFO - iteration 0, current learner lgbm
[flaml.automl.logger: 02-02 22:09:12] {2417} INFO - Estimated sufficient time budget=90272s. Estimated necessary time budget=90s.
[flaml.automl.logger: 02-02 22:09:12] {2466} INFO -  at 17.0s,	estimator lgbm's best error=0.3928,	best estimator lgbm's best error=0.3928
[flaml.automl.logger: 02-02 22:09:12] {2282} INFO - iteration 1, current learner lgbm
[flaml.automl.logger: 02-02 22:09:19] {2466} INFO -  at 24.5s,	estimator lgbm's best error=0.3928,	best estimator lgbm's best error=0.3928
[flaml.automl.logger: 02-0